# Staged Sentiment Analysis Training on Kaggle GPU
### Enterprise 3-Class Sentiment AI: Stage A (Sentiment140) + Stage B (TweetEval)

**Hardware & Settings Setup on Kaggle (Right Sidebar):**
1. **Accelerator:** Select **GPU T4 x2** (or **GPU P100**)
2. **Internet:** Toggle **Internet on** (required for downloads and HuggingFace)

*Zero manual file uploads required: Everything downloads directly via Kaggle's cloud network in 20 seconds.*

In [ ]:
# 1. Verify GPU Hardware
import torch
print("=" * 30 + " GPU DIAGNOSTICS " + "=" * 30)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM Capacity:  {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ Please select GPU T4 or P100 in the right sidebar under Accelerator!")
!nvidia-smi

In [ ]:
# 2. Clone Code & Setup Cleaned Datasets in Kaggle Cloud (~25 seconds total)
import os
%cd /kaggle/working

# Clone latest code from GitHub
!rm -rf AIproject
!git clone -b phase/2-data-model-audit https://github.com/HabtamuShewamene/AIproject.git /kaggle/working/AIproject
%cd /kaggle/working/AIproject

# Install dependencies
!pip install -q transformers>=4.46.0 datasets>=3.1.0 accelerate>=1.1.0 scikit-learn pandas seaborn matplotlib

# Download Sentiment140 directly into Kaggle (~10s)
!mkdir -p data/raw
if not os.path.exists("data/raw/training.1600000.processed.noemoticon.csv"):
    print("Downloading Sentiment140 directly from Stanford mirror (~10s)...")
    !wget -q --show-progress -O data/raw/trainingandtestdata.zip http://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip
    !unzip -q -o data/raw/trainingandtestdata.zip -d data/raw/

# Prepare Enterprise 80/10/10 Sentiment140 & TweetEval Benchmark splits
print("Executing enterprise data preparation & splitting...")
!python -m src.prepare_sentiment140
!python -m src.download_tweet_eval

# Verify all 6 cleaned splits are ready
!ls -lh data/processed/

## 3. Stage A: Binary Social Domain Adaptation
Fine-tunes base BERT on enterprise-cleaned Sentiment140 (1.23M train, 154k val, 154k test).
Runs with full `fp16` GPU acceleration on Kaggle's 9-hour execution window.

In [ ]:
# Run Stage A (Sentiment140 Domain Adaptation)
!python -m src.train_staged_sentiment --stage a

## 4. Stage B: Three-Class Fine-Tuning on TweetEval
Loads the Stage A adapted encoder, switches to 3 output classes (`negative`, `neutral`, `positive`), and fine-tunes on TweetEval in ~5-8 minutes.

In [ ]:
# Run Stage B (Three-Class Fine-Tuning)
!python -m src.train_staged_sentiment --stage b

## 5. Stage C: Strict Benchmark Evaluation & Visualization

In [ ]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

metrics_file = "results/staged_training_metrics.json" if os.path.exists("results/staged_training_metrics.json") else "/kaggle/working/AIproject/results/staged_training_metrics.json"
with open(metrics_file, "r", encoding="utf-8") as f:
    metrics = json.load(f)

report = metrics.get("tweet_eval_test", {})
print("=" * 60)
print("TWEETEVAL 3-CLASS TEST EVALUATION METRICS")
print("=" * 60)

classes = ["negative", "neutral", "positive"]
rows = []
for c in classes:
    if c in report:
        rows.append({
            "Class": c.capitalize(),
            "Precision": f"{report[c]['precision'] * 100:.2f}%",
            "Recall": f"{report[c]['recall'] * 100:.2f}%",
            "F1-Score": f"{report[c]['f1-score'] * 100:.2f}%",
            "Support": report[c]['support'],
        })

df_report = pd.DataFrame(rows)
display(df_report)

macro_f1 = report.get("macro avg", {}).get("f1-score", 0)
accuracy = report.get("accuracy", 0)
print(f"\nOverall Test Accuracy: {accuracy * 100:.2f}%")
print(f"Macro Average F1:     {macro_f1 * 100:.2f}%")
print("=" * 60)

## 6. Interactive Prediction Playground
Test your model on live sentences and inspect softmax confidence distributions.

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = "models/bert_sentiment" if os.path.exists("models/bert_sentiment") else "/kaggle/working/AIproject/models/bert_sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

id2label = {0: "negative", 1: "neutral", 2: "positive"}

def analyze_sentiment(text: str):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)[0]
    pred_id = int(torch.argmax(probs).item())
    pred_label = id2label[pred_id]
    confidence = float(probs[pred_id].item())
    
    print(f"Input Text:   '{text}'")
    print(f"Sentiment:    {pred_label.upper()} ({confidence * 100:.2f}% confidence)")
    print("Probabilities:")
    for i, label in id2label.items():
        bar = "█" * int(probs[i].item() * 25)
        print(f"  {label.capitalize():8}: {probs[i].item() * 100:5.1f}% | {bar}")
    print("-" * 50)

# Sample tests
analyze_sentiment("I waited 3 hours and nobody helped me. Terrible customer service.")
analyze_sentiment("The flight is scheduled to arrive at 5:30 PM tomorrow.")
analyze_sentiment("This product exceeded all my expectations, absolutely phenomenal!")

## 7. Package Trained Model for 1-Click Kaggle Output Download

In [ ]:
import os
if os.path.exists("models/bert_sentiment"):
    !zip -r /kaggle/working/bert_sentiment_trained.zip models/bert_sentiment/ results/
else:
    !zip -r /kaggle/working/bert_sentiment_trained.zip /kaggle/working/AIproject/models/bert_sentiment/ /kaggle/working/AIproject/results/
print("=" * 60)
print("🎉 MODEL PACKAGED SUCCESSFULLY!")
print("Download 'bert_sentiment_trained.zip' directly from the right Output tab in Kaggle!")
print("=" * 60)